# 1. Tiktok engaging project


# TEEs — TikTok Engage Estimator


---

## 1. Contexto e introducción

TikTok es hoy una de las plataformas de video corto con mayor volumen de contenido y consumo a nivel global. Para creadores, marcas y agencias, el alcance de un video (medido en reproducciones) es la métrica que define si una pieza "funcionó" o no. Sin embargo, ese resultado solo se conoce después de publicar: la decisión de qué grabar, con qué música, de qué duración, con qué hashtags y desde qué lugar se toma sin ninguna estimación cuantitativa del alcance esperado.

Este proyecto plantea construir **TEE (TikTok Engage Estimator)**, un modelo que estime el número de reproducciones (`play_count`) de un video usando exclusivamente información disponible **antes** de su publicación: características del video (duración, calidad), de la música, del texto descriptivo, de los hashtags, del punto de interés geográfico y de la configuración de la publicación (permitir duets, stitches, comentarios, etc.). No se usa ninguna métrica de engagement posterior (likes, comentarios, compartidos) como variable de entrada.

La pregunta de fondo no es solo si se puede predecir, sino **cuánto** del éxito de un video está determinado por decisiones que el creador controla antes de publicar, y cuánto queda en manos del algoritmo de recomendación, del timing y de factores no observables.

## 2. Antecedentes

La predicción de popularidad de contenido en línea es un problema estudiado desde hace más de una década. Los antecedentes más relevantes para este proyecto son:

- **Szabo y Huberman (2010)** mostraron que la popularidad final de contenido en Digg y YouTube se puede predecir con buena precisión a partir de su popularidad *temprana* (primeras horas). Este es el enfoque "a posteriori" clásico, y sirve como contraste: nuestro proyecto elimina deliberadamente esa señal temprana para evaluar el escenario más difícil, el de un video que todavía no existe en la plataforma.

- **Khosla, Das Sarma y Hamid (2014)** separaron explícitamente los factores *intrínsecos* del contenido (características visuales, texto) de los factores *sociales* (seguidores, contexto del usuario) en la predicción de popularidad de imágenes en Flickr. Encontraron que los factores sociales dominan, pero que el contenido intrínseco aporta señal predictiva no trivial. Nuestro problema es análogo, con la diferencia de que el dataset no incluye métricas del autor, por lo que el modelo se restringe casi por completo a factores intrínsecos y de configuración.

- **Chen et al. (2016)**, con *Micro Tells Macro*, abordaron la predicción de popularidad de micro-videos (Vine) combinando modalidades visual, acústica, textual y social mediante un esquema de aprendizaje transductivo. Es el trabajo de referencia sobre video corto y confirma que la modalidad textual y acústica (música) aportan información relevante, dos de las fuentes que sí tenemos en el dataset.

- **Ling, Blackburn, De Cristofaro y Stringhini (2022)**, en *Slapping Cats, Bopping Heads, and Oreo Shakes*, estudiaron específicamente los indicadores de viralidad en TikTok. Sobre 400 videos etiquetados manualmente, entrenaron clasificadores que alcanzaron un AUC de 0.93; el predictor más fuerte fue el número de seguidores, seguido de características de producción como la escala del plano. Este trabajo es la referencia más cercana en plataforma, pero opera a una escala tres órdenes de magnitud menor y depende de etiquetado manual; nuestro proyecto explora qué se puede lograr con metadatos estructurados a escala de millones de registros.

En conjunto, la literatura sugiere que (a) la popularidad se predice bien cuando hay señal social o temprana, (b) sin esa señal el problema es sustancialmente más difícil, y (c) aun así el contenido intrínseco aporta información. TEE se ubica en el escenario (b)-(c).

## 3. Objetivos

### Objetivo general

Desarrollar y evaluar un modelo de regresión que estime el número de reproducciones de un video de TikTok a partir de metadatos disponibles a priori, y cuantificar qué tan predecible es el alcance de un video antes de su publicación.

### Objetivos específicos

1. Construir un pipeline reproducible de preparación de datos sobre el dataset TikTok-10M que seleccione, tipifique, impute y limpie las variables a priori definidas para el proyecto.
2. Caracterizar mediante análisis exploratorio la distribución de `play_count` y su relación con las variables booleanas, nominales y numéricas del conjunto de features.
3. Entrenar y comparar al menos tres familias de modelos de regresión (lineal regularizada, basado en árboles/boosting y una alternativa a definir) sobre una transformación adecuada del target.
4. Evaluar el desempeño con métricas apropiadas para un target de cola pesada (por ejemplo, RMSLE, MAE en escala log y correlación de Spearman) y contrastarlo con baselines triviales (media, mediana, mediana por categoría).
5. Identificar las variables con mayor poder predictivo y discutir qué decisiones controlables por el creador tienen efecto medible sobre el alcance estimado.

## 4. Planteamiento del problema

Un creador que está por publicar un video no tiene forma de saber si alcanzará mil o diez millones de reproducciones. Las herramientas de analítica de TikTok solo muestran resultados después del hecho, y la intuición sobre "qué funciona" es anecdótica y difícil de transferir.

Formalmente: dado un vector de características $x$ observable en el momento de la publicación (configuración del video, música, texto, hashtags, ubicación, duración y calidad), se busca una función $\hat{f}(x)$ que aproxime $\mathbb{E}[\log(1 + \text{play\_count}) \mid x]$, o equivalentemente un estimador del orden de magnitud del alcance.

Las dificultades específicas del problema son:

- **Distribución del target.** `play_count` sigue una distribución de cola muy pesada: la gran mayoría de los videos tiene pocas reproducciones y una fracción mínima concentra el volumen. Esto obliga a transformar el target y a elegir métricas robustas.
- **Ausencia de señal social.** El dataset no expone seguidores ni historial del autor, que la literatura identifica como el predictor dominante. El modelo trabaja en el escenario más adverso.
- **Alta cardinalidad.** Variables como `music_id`, `poi_id`, `challenges` y `desc` tienen miles o millones de valores únicos; requieren encoding cuidadoso (target encoding, frecuencia, embeddings o extracción de features textuales).
- **Sesgo de muestreo.** El dataset contiene contenido *trending* con POI en Estados Unidos durante la primavera de 2025, lo que limita la generalización y puede inflar el desempeño aparente.

La hipótesis de trabajo es que las variables a priori explican una fracción modesta pero estadísticamente significativa de la varianza de `log(play_count)`, suficiente para discriminar órdenes de magnitud, pero no para predecir valores puntuales con precisión.

## 5. Descripción y justificación de los datos

### Fuente

Se utiliza el dataset **TikTok-10M** publicado por *The Data Company* en Hugging Face (`The-data-company/TikTok-10M`). Contiene alrededor de 10 millones de publicaciones de TikTok (≈6.65 M en la versión convertida a Parquet), con metadatos de la publicación, estadísticas de engagement, datos del punto de interés (POI), datos de la música y datos del video. La licencia se reporta como "other"; el dataset se declara construido únicamente con datos públicos y para uso en investigación.

### Variables utilizadas

Se descartan las columnas que son métricas de engagement posterior (`digg_count`, `comment_count`, `share_count`, `collect_count`), las URLs, los identificadores de usuario y avatares, y las columnas redundantes o degeneradas (`stitch_display`, `duet_display`, `city`, `poi_tt_type_name_super`, `poi_tt_type_code`, `country_code`). El conjunto final es:

| Grupo | Variables | Justificación |
|---|---|---|
| **Booleanas** (9) | `duet_enabled`, `is_ad`, `item_mute`, `item_control_can_repost`, `official_item`, `original_item`, `share_enabled`, `stitch_enabled`, `music_original` | Configuración de la publicación que el creador decide explícitamente; afectan la capacidad del video de propagarse (duet, stitch, repost, share). |
| **Nominales** (16) | `desc`, `address`, `poi_name`, `city_code`, `poi_category`, `poi_tt_type_name_medium`, `poi_tt_type_name_tiny`, `challenges`, `music_id`, `music_title`, `music_album`, `duet_info_duet_from_id`, `music_author_name`, `poi_id`, `diversification_id`, `item_comment_status` | Contenido textual, hashtags, música y ubicación: las modalidades que la literatura identifica como informativas y que se fijan antes de publicar. |
| **Numéricas** (3) | `music_duration`, `vq_score`, `duration` | Duración del video y de la música, y puntaje de calidad de video; características de producción controlables. |
| **Target** | `play_count` | Número de reproducciones; métrica principal de alcance. |

### Justificación

El dataset es adecuado porque (1) está a una escala que permite modelos con alta cardinalidad sin sobreajuste inmediato, (2) incluye explícitamente las modalidades a priori (texto, música, POI, configuración) sin necesidad de descargar los videos, y (3) su documentación declara los sesgos (temporal, geográfico y de contenido), lo que permite acotar las conclusiones. Su principal limitación es la ausencia de métricas del autor, que se asume deliberadamente como parte del planteamiento.

## 6. Síntesis de los hallazgos del EDA y de la preparación

*Análisis realizado sobre una muestra de ~2 M registros (`sample_data.parquet`) del dataset TikTok-10M.*

### Calidad de datos

- **Tipos de dato incorrectos.** Varias variables están almacenadas con un tipo que no corresponde a su semántica: las banderas como `is_ad` vienen como cadenas `t`/`f` en lugar de booleanos, y los campos temporales (`create_time` y similares) vienen como cadena en lugar de datetime.
- **Valores faltantes.** Individualmente, la mayoría de las variables tiene una proporción marginal de nulos. Sin embargo, si se exige que un registro no tenga ningún nulo, queda **menos del 1 % de los datos**, por lo que es indispensable imputar. Se asume que basta con imputar las numéricas, ya que muchas de las columnas con nulos (`address`, `user_avatar_*`, `url`, etc.) se descartarán de todas formas por privacidad, leakage o irrelevancia.
- **Duplicados.** La proporción de `id` duplicados es muy pequeña; eliminarlos es una decisión aceptable.

### Target (`play_count`)

- Distribución con **varianza y desviación estándar altas, y sesgo fuerte** (cola pesada), visible en boxplot e histograma en escala log.
- Conclusión directa: se deberá modelar una transformación del target, $\ln(y)$ o una potencia $y^\lambda$ (Box-Cox), porque en escala original quedan demasiados outliers.
- Se separaron las `POST_STATISTICS_FEATURES` (`share_count`, `play_count`, `digg_count`, `comment_count`, `collect_count`) de las numéricas porque son material del target, no features.

### Variables booleanas

- Las nueve dummies están **muy desbalanceadas**; ninguna se acerca al 50 %. Esto puede ser un problema en el entrenamiento y debe considerarse en el preprocesamiento.
- Al comparar la distribución de `play_count` (escala log, violin plots) entre `True` y `False` para cada dummy, **las distribuciones son muy similares**. Las diferencias más marcadas aparecen justamente en las variables más desbalanceadas, lo que las hace poco confiables. Resultado poco alentador de forma preliminar.

### Variables nominales

- **`city` vs `city_code`.** `city` tiene más de la mitad de valores nulos; `city_code` apenas tiene. Crosstab e histogramas de ciudades distintas por código muestran que ambas son colineales a efectos prácticos (con una proporción pequeña de `city_code` asociados a más de una ciudad). Decisión: **dropear `city` y trabajar con `city_code`**.
- **Colinealidad en POI.** Los pares `poi_category`/`poi_tt_type_name_super` y `poi_tt_type_name_tiny`/`poi_tt_type_code` son colineales (mismo nivel de granularidad, crosstabs casi diagonales). Decisión: **conservar una de cada par** (`poi_category` y `poi_tt_type_name_tiny`).
- **`country_code`** es constante o casi constante en la muestra: **inútil**, se descarta.
- **Cardinalidad.** Solo unas ~5 variables tienen granularidad baja como para estimar medias condicionales directamente (desde `poi_category` hasta `poi_tt_type_code`, con un máximo de ~364 clases para 2 M de filas). Todo lo que supera ~1 000 valores distintos necesita feature engineering adicional.
- **`music_title` vs `music_id`.** Hay una diferencia grande de cardinalidad entre ambos; la hipótesis es que muchas canciones distintas comparten título. Pendiente de verificar.
- **Distribuciones de frecuencia (top 40 + ojiva).** `desc` tiene una clase dominante pero demasiada variedad para tratarse como categórica. `address` muestra un codo de Pareto más definido, lo que permite fijar un umbral y agrupar la cola en una etiqueta `other`. El mismo tratamiento parece viable para `city_code`, `poi_tt_type_name_medium`, `poi_tt_type_name_tiny` y `diversification_id`.
- Las variables de texto libre (`desc`, `challenges`) se dejaron para una etapa posterior porque requieren técnicas de feature engineering más sofisticadas.

### Variables numéricas

- `duet_display` y `stitch_display` toman **un solo valor**: se descartan.
- `duration` y `music_duration` tienen **correlación alta** entre sí; hay que cuidarlo si se usan modelos lineales con interpretación sobre los pesos, aunque de entrada no parece que un modelo lineal vaya a ajustar bien.
- `vq_score` tiene una masa de valores en 0; hay oportunidad de discretizar o crear una bandera `no_vq` para tratar esos casos.
- Scatter plots y pairplot contra `play_count` (escala log) **no muestran relación aparente** entre las numéricas y el target.

### Decisiones de preparación derivadas del EDA

1. Dropear `stitch_display`, `duet_display`, `city`, `poi_tt_type_name_super`, `poi_tt_type_code`, `country_code`.
2. Quedarse con 9 booleanas, 16 nominales, 3 numéricas y el target `play_count`.
3. Castear booleanas a `bool` e identificadores (`*_id`) a `object`/`category`.
4. Imputar numéricas según la estrategia recomendada por la literatura.
5. Dropear la cantidad marginal de nulos restantes.

### Riesgos éticos y sesgos

- No se usan variables temporales (`create_time`) por posible leakage.
- No se usan variables de información del usuario (`user_id`, avatares, `user_verified`, `user_tt_seller`) por privacidad y leakage.

### Conclusión general del EDA

Dataset relativamente limpio, con muchos metadatos y abundancia de variables nominales o de texto de alta granularidad. Pocas features numéricas y sin relación aparente con el target. El target es una variable de alta varianza, alta desviación estándar y bastante sesgada. La preparación que sigue es deliberadamente conservadora.




### Preparación de datos

El pipeline de preparación se implementó en un notebook de marimo con pandas, factorizado en funciones encadenables (`select_columns → cast_types → impute_numericals → drop_residual_nulls`). Los pasos aplicados fueron:

1. **Selección de variables.** Eliminación de las columnas descartadas y restricción al conjunto de 28 features + target.
2. **Tipificación.** Booleanas a dtype `boolean` nullable (evitando la conversión silenciosa de nulos a `True`) y a `bool` nativo al final; identificadores (`*_id`) a `category`; nominales restantes a `string`; numéricas y target a numérico con coerción.
3. **Imputación de numéricas.** Estrategia univariada decidida por columna según la asimetría de la distribución: mediana cuando $|\text{skew}| > 1$, media en caso contrario (Little & Rubin, 2019).
4. **Eliminación de nulos residuales.** Eliminación por fila de los registros con nulos restantes tras la imputación, dado que representan una fracción marginal.

en esta instancia la preparacion y eda de los datos se realizo sobre un subconjunto de 2_000_000 de filas, de las cuales se desecharon cerca de un 15% por nulos no imputables

### Hallazgos del análisis exploratorio


## 7. Conclusiones parciales y siguientes pasos

### Conclusiones parciales


Como mencionamos al cerrar el EDA el dataset contiene relativamente pocas variables numericas y que no parecen tener una relacion sencilla de aprender por lo que se deberia de hacer muchop enfasis es en el data engineering de las variables de texto, ya que estas pueden contener keywords para describir de mejor forma la naturaleza del video y quizas entonces modelar de mejor forma las visualizaciones
.

### Siguientes pasos

1. **Ingeniería de features.** Longitud y conteo de hashtags en `desc`/`challenges`; frecuencia o target encoding para `music_id`, `poi_id`, `poi_category`; extracción de features textuales (TF-IDF o embeddings) para `desc`.
2. **Definición del protocolo de evaluación.** Split estratificado por cuantiles del target (o temporal por `create_time` si se recupera esa columna), baselines triviales y métricas: RMSLE, MAE en log, Spearman.
3. **Modelado.** Regresión lineal regularizada (Ridge/Lasso) como baseline interpretable; gradient boosting (LightGBM/XGBoost) como modelo principal; evaluación de una formulación alternativa como clasificación ordinal por orden de magnitud.
4. **Interpretación.** Importancia de variables (permutación, SHAP) y análisis de qué decisiones controlables tienen efecto medible.
5. **Discusión de límites.** Cuantificar el techo de predictibilidad sin señal social y contrastar con lo reportado en la literatura.

## 8. Referencias

- Chen, J., Song, X., Nie, L., Wang, X., Zhang, H., & Chua, T.-S. (2016). Micro Tells Macro: Predicting the Popularity of Micro-Videos via a Transductive Model. *Proceedings of the 24th ACM International Conference on Multimedia (MM '16)*, 898–907. https://doi.org/10.1145/2964284.2964314
- Khosla, A., Das Sarma, A., & Hamid, R. (2014). What Makes an Image Popular? *Proceedings of the 23rd International Conference on World Wide Web (WWW '14)*, 867–876. https://doi.org/10.1145/2566486.2567996
- Ling, C., Blackburn, J., De Cristofaro, E., & Stringhini, G. (2022). Slapping Cats, Bopping Heads, and Oreo Shakes: Understanding Indicators of Virality in TikTok Short Videos. *Proceedings of the 14th ACM Web Science Conference (WebSci '22)*, 164–173. https://doi.org/10.1145/3501247.3531551
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3.ª ed.). Wiley.
- Szabo, G., & Huberman, B. A. (2010). Predicting the Popularity of Online Content. *Communications of the ACM, 53*(8), 80–88. https://doi.org/10.1145/1787234.1787254
- The Data Company. (2025). *TikTok-10M: A Large-Scale Short Video Dataset for Video Understanding* [Dataset]. Hugging Face. https://huggingface.co/datasets/The-data-company/TikTok-10M